# Repro Paper 2: *Edge-AI Cost-Sensitive BiGRU+WBCE* (H. C. Phap, 2026) & *ExplainableDetector RoBERTa/DistilBERT* (Uddin et al., 2024)
**Objective:** Strict 1:1 reproduction of the two foundational SMS spam detection papers on **their exact original public dataset: UCI SMS Spam Collection ($N = 5,574$ messages: `4,827` ham [86.6%] and `747` spam [13.4%])**:
1. **Uddin et al. (2024, `M034` *ExplainableDetector*, Table 2, 3, 4)**:
   - **Original Split:** 80% Train (`4,457` messages) / 20% Test (`1,115` messages: `965` ham, `150` spam).
   - **Original Hyperparameters:** `roberta-base` & `distilbert-base-uncased`, `AdamW`, `lr = 2e-5`, `batch_size = 32` (train) / `64` (test), `epochs = 5`, plus classical ML baselines (`XGBoost`, `SVM`, `Random Forest`).
   - **Published Metrics on Imbalanced UCI Test Set ($N=1,115$):**
     - `RoBERTa`: Accuracy = **`99.37%`**, Spam Precision = **`0.99`**, Spam Recall = **`0.97`**, Spam F1 = **`0.98`** (Confusion Matrix: `TN=963, FP=2, FN=5, TP=145`).
     - `DistilBERT`: Accuracy = **`99.28%`**, Spam Precision = **`0.97`**, Spam Recall = **`0.97`**, Spam F1 = **`0.97`** (`TN=961, FP=4, FN=4, TP=146`).
2. **H. C. Phap (2026, `M041` *Edge-AI enabled smishing and suspicious SMS detection: a cost-sensitive and lightweight deep learning architecture*)**:
   - **Original Dataset:** UCI SMS Spam Collection ($N = 5,574$).
   - **Original Models & Published Hardware Footprint:**
     - `TF-IDF + Logistic Regression`
     - `BiGRU + MaskedGlobalMaxAvgPool + Weighted Binary Cross-Entropy (WBCE)`: Serialized footprint = **`5.21 MB`**, CPU latency = **`0.25 ms/msg`**.
     - `DistilBERT`: Serialized footprint = **`256.02 MB`**, CPU latency = **`40.78 ms/msg`**, Recall = **`0.9866`**, Average Precision = **`0.9930`**.

In [ ]:
# Cell 1: Install dependencies on Kaggle (Turn ON Internet & GPU T4/P100)
!pip install -q transformers datasets accelerate scikit-learn xgboost lime pandas numpy torch

In [ ]:
# Cell 2: Load the Exact Original UCI SMS Spam Collection (N = 5,574) & 80/20 Stratified Split
import os
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import LinearSVC
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
    confusion_matrix,
)

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"

# Load UCI SMS Spam Collection from HuggingFace (ucirvine/sms_spam)
uci_ds = load_dataset("ucirvine/sms_spam", split="train")
df = pd.DataFrame(uci_ds)
df["text"] = df["sms"].astype(str)
df["label"] = df["label"].astype(int)  # 0 = ham (4,827), 1 = spam (747)

print(f"Total UCI SMS messages: {len(df)} | Ham (0): {(df['label']==0).sum()} | Spam (1): {(df['label']==1).sum()}")

# Exact 80/20 split matching Uddin et al. (2024) Table 1 (4,459 train / 1,115 test: 965 ham, 150 spam)
train_df, test_df = train_test_split(
    df[["text", "label"]],
    test_size=1115,
    stratify=df["label"],
    random_state=42,
)
print(f"Train split: {len(train_df)} | Test split: {len(test_df)} (Ham={(test_df['label']==0).sum()}, Spam={(test_df['label']==1).sum()})")

In [ ]:
# Cell 3: Reproduce Classical ML Baselines (TF-IDF + LR, SVM, Random Forest, XGBoost)
vec = TfidfVectorizer(max_features=5000, ngram_range=(1, 2), sublinear_tf=True)
X_train = vec.fit_transform(train_df["text"])
X_test = vec.transform(test_df["text"])
y_train = train_df["label"].values
y_test = test_df["label"].values

ml_models = {
    "TF-IDF + Logistic Regression (HC Phap 2026)": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42),
    "TF-IDF + Linear SVM (Uddin et al. 2024)": LinearSVC(random_state=42),
    "TF-IDF + Random Forest (Uddin et al. 2024)": RandomForestClassifier(n_estimators=200, random_state=42),
    "TF-IDF + XGBoost (Uddin et al. 2024)": XGBClassifier(n_estimators=200, eval_metric="logloss", random_state=42),
}

results_list = []

for name, clf in ml_models.items():
    clf.fit(X_train, y_train)
    t0 = time.perf_counter()
    preds = clf.predict(X_test)
    cpu_ms = ((time.perf_counter() - t0) * 1000.0) / len(y_test)
    if hasattr(clf, "predict_proba"):
        probs = clf.predict_proba(X_test)[:, 1]
    else:
        probs = clf.decision_function(X_test)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    results_list.append({
        "Model": name,
        "Paper Acc (%)": "98.48 (XGB) / 98.20 (SVM)",
        "Kaggle Acc (%)": round(accuracy_score(y_test, preds) * 100, 2),
        "Spam Precision": round(precision_score(y_test, preds), 4),
        "Spam Recall": round(recall_score(y_test, preds), 4),
        "Spam F1": round(f1_score(y_test, preds), 4),
        "Avg Precision (AP)": round(average_precision_score(y_test, probs), 4),
        "Confusion (TN,FP,FN,TP)": f"({tn}, {fp}, {fn}, {tp})",
        "CPU Latency (ms/msg)": round(cpu_ms, 4),
        "Size (MB)": "N/A (<1 MB)",
    })

pd.DataFrame(results_list)

In [ ]:
# Cell 4: Reproduce H. C. Phap (2026) — Lightweight BiGRU + Masked Global Avg/Max Pool + Weighted BCE (WBCE)
from collections import Counter

# Build word vocabulary (max_vocab = 15,000)
counter = Counter()
for txt in train_df["text"]:
    counter.update(txt.lower().split())
vocab = {"<PAD>": 0, "<UNK>": 1}
for word, _ in counter.most_common(14998):
    vocab[word] = len(vocab)

def encode_text(txt, max_len=100):
    tokens = [vocab.get(w, 1) for w in txt.lower().split()[:max_len]]
    mask = [1.0] * len(tokens)
    pad_len = max_len - len(tokens)
    return tokens + [0] * pad_len, mask + [0.0] * pad_len

class SMSBiGRUDataset(Dataset):
    def __init__(self, texts, labels):
        self.data = [encode_text(t) for t in texts]
        self.labels = labels.values.astype(np.float32)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        ids, mask = self.data[i]
        return torch.tensor(ids, dtype=torch.long), torch.tensor(mask, dtype=torch.float32), torch.tensor(self.labels[i], dtype=torch.float32)

class CostSensitiveBiGRU(nn.Module):
    def __init__(self, vocab_size, embed_dim=128, hidden_dim=128, dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.bigru = nn.GRU(embed_dim, hidden_dim, batch_first=True, bidirectional=True)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim * 4, 1)

    def forward(self, input_ids, mask):
        emb = self.dropout(self.embedding(input_ids))
        out, _ = self.bigru(emb)  # (B, L, 2*H)
        mask_unsq = mask.unsqueeze(-1)
        # Masked Global Average + Max Pooling (HC Phap 2026 architecture)
        avg_pool = (out * mask_unsq).sum(dim=1) / mask_unsq.sum(dim=1).clamp(min=1e-6)
        max_pool = (out.masked_fill(mask_unsq == 0, -1e9)).max(dim=1)[0]
        feat = self.dropout(torch.cat([avg_pool, max_pool], dim=-1))
        return self.fc(feat).squeeze(-1)

set_seed(42)
train_loader = DataLoader(SMSBiGRUDataset(train_df["text"], train_df["label"]), batch_size=32, shuffle=True)
test_loader = DataLoader(SMSBiGRUDataset(test_df["text"], test_df["label"]), batch_size=64, shuffle=False)

# Compute exact inverse class frequency positive weight for WBCE (N_ham / N_spam = 4827 / 747 ≈ 6.46)
pos_weight_val = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight_val], device=device))

bigru_model = CostSensitiveBiGRU(len(vocab)).to(device)
optimizer = torch.optim.AdamW(bigru_model.parameters(), lr=1e-3, weight_decay=1e-4)

for epoch in range(8):
    bigru_model.train()
    for ids, mask, lbls in train_loader:
        ids, mask, lbls = ids.to(device), mask.to(device), lbls.to(device)
        optimizer.zero_grad()
        logits = bigru_model(ids, mask)
        loss = criterion(logits, lbls)
        loss.backward()
        optimizer.step()

# Save and measure serialized model footprint (MB) and CPU latency (ms/msg)
torch.save(bigru_model.state_dict(), "hc_phap_bigru_wbce.pt")
bigru_size_mb = os.path.getsize("hc_phap_bigru_wbce.pt") / (1024 * 1024)

bigru_cpu = bigru_model.to("cpu").eval()
all_probs = []
t0 = time.perf_counter()
with torch.no_grad():
    for ids, mask, _ in test_loader:
        logits = bigru_cpu(ids, mask)
        all_probs.extend(torch.sigmoid(logits).numpy().tolist())
bigru_cpu_ms = ((time.perf_counter() - t0) * 1000.0) / len(y_test)

all_probs = np.array(all_probs)
all_preds = (all_probs >= 0.5).astype(int)
tn, fp, fn, tp = confusion_matrix(y_test, all_preds).ravel()

results_list.append({
    "Model": "BiGRU + MaskedPool + WBCE (HC Phap 2026)",
    "Paper Acc (%)": "Edge-AI (5.21 MB / 0.25 ms)",
    "Kaggle Acc (%)": round(accuracy_score(y_test, all_preds) * 100, 2),
    "Spam Precision": round(precision_score(y_test, all_preds), 4),
    "Spam Recall": round(recall_score(y_test, all_preds), 4),
    "Spam F1": round(f1_score(y_test, all_preds), 4),
    "Avg Precision (AP)": round(average_precision_score(y_test, all_probs), 4),
    "Confusion (TN,FP,FN,TP)": f"({tn}, {fp}, {fn}, {tp})",
    "CPU Latency (ms/msg)": round(bigru_cpu_ms, 4),
    "Size (MB)": f"{bigru_size_mb:.2f} MB (Paper: 5.21 MB)",
})

In [ ]:
# Cell 5: Reproduce DistilBERT & RoBERTa (Uddin et al. 2024 Table 2-4 & HC Phap 2026)
from datasets import Dataset as HFDataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, DataCollatorWithPadding

TRANSFORMER_MODELS = [
    ("distilbert-base-uncased", "99.28% (TN=961,FP=4,FN=4,TP=146)", "256.02 MB / 40.78 ms"),
    ("roberta-base", "99.37% (TN=963,FP=2,FN=5,TP=145)", "498.00 MB"),
]

for hf_name, paper_acc_str, paper_size_str in TRANSFORMER_MODELS:
    print("=" * 80)
    print(f"Training {hf_name} on UCI SMS 80/20 split (lr=2e-5, bs=32, epochs=5)...")
    set_seed(42)
    tokenizer = AutoTokenizer.from_pretrained(hf_name)
    model = AutoModelForSequenceClassification.from_pretrained(hf_name, num_labels=2)

    def tok_fn(batch):
        return tokenizer(batch["text"], truncation=True, max_length=128)

    tr_ds = HFDataset.from_pandas(train_df.reset_index(drop=True)).map(tok_fn, batched=True)
    te_ds = HFDataset.from_pandas(test_df.reset_index(drop=True)).map(tok_fn, batched=True)

    args = TrainingArguments(
        output_dir=f"./uci_{hf_name}",
        learning_rate=2e-5,               # Exact Uddin et al. 2024 Table 1 lr
        per_device_train_batch_size=32,   # Exact Uddin et al. 2024 Table 1 train batch size
        per_device_eval_batch_size=64,    # Exact Uddin et al. 2024 Table 1 test batch size
        num_train_epochs=5,               # Exact Uddin et al. 2024 Table 1 epochs
        weight_decay=0.01,
        fp16=torch.cuda.is_available(),
        report_to="none",
        save_strategy="no",
        seed=42,
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=tr_ds,
        eval_dataset=te_ds,
        processing_class=tokenizer,
        data_collator=DataCollatorWithPadding(tokenizer),
    )
    trainer.train()

    pred_out = trainer.predict(te_ds)
    logits = pred_out.predictions
    probs = torch.softmax(torch.tensor(logits), dim=-1)[:, 1].numpy()
    preds = np.argmax(logits, axis=-1)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()

    # Measure serialized footprint & CPU latency on 100 test messages
    ckpt_file = f"{hf_name}_uci.pt"
    torch.save(model.state_dict(), ckpt_file)
    size_mb = os.path.getsize(ckpt_file) / (1024 * 1024)

    model_cpu = model.to("cpu").eval()
    sample_texts = test_df["text"].iloc[:100].tolist()
    t0 = time.perf_counter()
    with torch.no_grad():
        for txt in sample_texts:
            inp = tokenizer(txt, return_tensors="pt", truncation=True, max_length=128)
            _ = model_cpu(**inp)
    cpu_ms = ((time.perf_counter() - t0) * 1000.0) / len(sample_texts)

    results_list.append({
        "Model": f"{hf_name} (Uddin 2024 / HC Phap 2026)",
        "Paper Acc (%)": paper_acc_str,
        "Kaggle Acc (%)": round(accuracy_score(y_test, preds) * 100, 2),
        "Spam Precision": round(precision_score(y_test, preds), 4),
        "Spam Recall": round(recall_score(y_test, preds), 4),
        "Spam F1": round(f1_score(y_test, preds), 4),
        "Avg Precision (AP)": round(average_precision_score(y_test, probs), 4),
        "Confusion (TN,FP,FN,TP)": f"({tn}, {fp}, {fn}, {tp})",
        "CPU Latency (ms/msg)": round(cpu_ms, 2),
        "Size (MB)": f"{size_mb:.2f} MB (Paper: {paper_size_str})",
    })

final_uci_df = pd.DataFrame(results_list)
final_uci_df.to_csv("Repro_Paper2_UCI_SMS_Comparison.csv", index=False)
print("\n=== SIDE-BY-SIDE COMPARISON WITH UDDIN ET AL. (2024) & H. C. PHAP (2026) ON UCI SMS (N=5,574) ===")
display(final_uci_df)